# Assignment 04 — Heart Disease (Dataset: heart_disease_uci.csv)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import joblib
import os

print("imports ok")


def detect_target(df):
    # Common target name options in heart disease datasets
    candidates = ['target', 'diagnosis', 'num', 'heart_disease', 'condition']
    for c in candidates:
        if c in df.columns:
            return c
    # fall back to last column if it's numeric/binary-like
    last = df.columns[-1]
    if df[last].nunique() <= 5:
        return last
    return None


def ensure_models_dir():
    os.makedirs('models', exist_ok=True)



In [ ]:
# Load dataset

df = pd.read_csv('heart_disease_uci.csv')
print('shape:', df.shape)
df.head()

# Basic EDA
print('\ninfo:')
print(df.info())
print('\ndescribe:')
print(df.describe())

print('\nMissing values per column:')
print(df.isna().sum())

# Detect target
target_col = detect_target(df)
print('\nDetected target column:', target_col)

if target_col is None:
    raise ValueError('Could not detect target column automatically; please edit the notebook to set the target column name.')

# Preprocessing (simple)
X = df.drop(columns=[target_col])
y = df[target_col]

# Fill numeric NaNs with median, categorical with mode
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

from sklearn.impute import SimpleImputer
num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')

if len(num_cols):
    X[num_cols] = num_imputer.fit_transform(X[num_cols])
if len(cat_cols):
    X[cat_cols] = cat_imputer.fit_transform(X[cat_cols])

# One-hot encode categoricals if any
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# Scale numeric features
scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=(y if y.nunique()<=10 else None))

# Baseline: Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)
print('\nLogistic Regression metrics:')
print('accuracy', accuracy_score(y_test, y_pred))
print('precision', precision_score(y_test, y_pred, average='binary' if y.nunique()==2 else 'macro', zero_division=0))
print('recall', recall_score(y_test, y_pred, average='binary' if y.nunique()==2 else 'macro', zero_division=0))
print('f1', f1_score(y_test, y_pred, average='binary' if y.nunique()==2 else 'macro', zero_division=0))

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
print('\nRandom Forest metrics:')
print('accuracy', accuracy_score(y_test, y_pred_rf))
print('precision', precision_score(y_test, y_pred_rf, average='binary' if y.nunique()==2 else 'macro', zero_division=0))
print('recall', recall_score(y_test, y_pred_rf, average='binary' if y.nunique()==2 else 'macro', zero_division=0))
print('f1', f1_score(y_test, y_pred_rf, average='binary' if y.nunique()==2 else 'macro', zero_division=0))

# Feature importances
importances = None
if hasattr(rf, 'feature_importances_'):
    importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
    print('\nTop features:')
    print(importances.head(10))

# Save best model (choose rf here)
ensure_models_dir()
joblib.dump(rf, 'models/heart_disease_model.joblib')
print('\nSaved RandomForest model to models/heart_disease_model.joblib')
